# Сегментация клиентской базы: k-means

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import psycopg2
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

MY_LOGIN = ''
MY_PASSWORD = ''

conn = psycopg2.connect(
    host='',
    port=,
    dbname='',
    user=MY_LOGIN,
    password=MY_PASSWORD,
    sslmode='require'
)

def sql(query):
    """Выполнить SQL и вернуть результат как DataFrame."""
    return pd.read_sql(query, conn)

print('Подключено')

RANDOM_STATE = 42
pd.set_option('display.width', 160, 'display.max_columns', 40)
np.random.seed(0)

## 1. Загрузка данных
Данные загружаются напрямую из PostgreSQL через подключение `psycopg2` и функцию `sql()`.

In [ ]:
tables = {
    'customers':    'seg_customers',
    'customers360': 'seg_customers_360',
    'products':     'seg_products',
    'revenue':      'seg_revenue_monthly',
    'balances':     'seg_balances',
    'rfm_old':      'seg_rfm_old',
    'txn':          'seg_transactions',
}

for name, table in tables.items():
    n = sql(f'SELECT COUNT(*) AS n FROM {table}').iloc[0, 0]
    print(f'{name:13s}: {n:>10,} строк')

# 2. Санити-чек и выбор источника операций

In [ ]:
print(sql("""
SELECT min(txn_ts)::date AS d_min, max(txn_ts)::date AS d_max,
       count(*) AS rows, count(DISTINCT txn_id) AS uniq_id,
       count(DISTINCT customer_id) AS seg_customers
FROM seg_transactions""").to_string(index=False))

# структура каналов/направлений: все ATM = наличные out, все p2p = входящие переводы
print()
print(sql("SELECT channel, direction, count(*) n FROM seg_transactions GROUP BY 1,2 ORDER BY 1").to_string(index=False))

In [ ]:
# помесячная динамика: рост базы ) + сезонные всплески (мар/сен/дек)
mon = sql("""SELECT to_char(txn_ts, 'YYYY-MM') m, count(*) n, count(DISTINCT customer_id) cust
                      FROM seg_transactions GROUP BY 1 ORDER BY 1""")
fig, ax = plt.subplots(1,2, figsize=(13,3.2))
ax[0].plot(mon['m'], mon['n']/1e3);   ax[0].set_title('Операций в месяц, тыс'); ax[0].tick_params(axis='x', rotation=90)
ax[1].plot(mon['m'], mon['cust']/1e3);ax[1].set_title('Активных клиентов, тыс');ax[1].tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.show()

## 3. Окно признаков

Берём **2025 календарный год (12 мес)** по клиентам, активным в этом окне.

- База растёт по когортам → накопленные за весь период счётчики мерили бы стаж, а не поведение.
  Поэтому фиксированное недавнее окно + нормировка на активные месяцы и доли.
- 12 месяцев гасят сезонность (мар/сен/дек).
- Недавнее окно = текущее поведение, на котором и принимаются решения.

In [ ]:
WIN_START = '2025-01-01'
feat = sql(f"""
WITH win AS (SELECT * FROM seg_transactions WHERE txn_ts >= DATE '{WIN_START}'),
tx AS (
  SELECT customer_id,
    count(*) txn_cnt,
    count(DISTINCT date_trunc('month',txn_ts)) active_months,
    count(*) FILTER(WHERE direction='out') out_cnt,
    count(*) FILTER(WHERE direction='out' AND channel='atm')  atm_cnt,
    count(*) FILTER(WHERE direction='out' AND channel='ecom') ecom_cnt,
    count(*) FILTER(WHERE direction='in'  AND channel='p2p')  p2p_in_cnt,
    sum(amount_uzs) FILTER(WHERE direction='out') out_amt
  FROM win GROUP BY customer_id),
bal  AS (SELECT customer_id, avg(avg_balance_uzs) avg_balance
         FROM seg_balances WHERE month_start >= DATE '{WIN_START}' GROUP BY 1),
rev  AS (SELECT customer_id, sum(net_revenue_uzs) net_rev_12mo, sum(atm_cost_uzs) atm_cost_12mo
         FROM seg_revenue_monthly  WHERE month_start >= DATE '{WIN_START}' GROUP BY 1),
prod AS (SELECT customer_id, count(*) n_products FROM seg_products GROUP BY 1)
SELECT c.customer_id, c.region,
       EXTRACT(YEAR FROM age(DATE '2025-12-31', c.birth_date))::int age,
       tx.txn_cnt, tx.active_months, tx.out_cnt,
       tx.txn_cnt::double precision/NULLIF(tx.active_months,0)              txn_per_month,
       tx.atm_cnt::double precision /NULLIF(tx.out_cnt,0)                   atm_share,
       tx.ecom_cnt::double precision/NULLIF(tx.out_cnt,0)                   ecom_share,
       tx.out_amt::double precision /NULLIF(tx.out_cnt,0)                   avg_ticket,
       tx.p2p_in_cnt::double precision/NULLIF(tx.txn_cnt,0)                 inflow_share,
       COALESCE(bal.avg_balance,0)  avg_balance,
       COALESCE(rev.net_rev_12mo,0) net_rev_12mo,
       COALESCE(rev.atm_cost_12mo,0) atm_cost_12mo,
       COALESCE(prod.n_products,0)  n_products,
       cc.crm_segment, cc.turnover_total
FROM seg_customers c
JOIN tx USING(customer_id)
LEFT JOIN bal USING(customer_id) LEFT JOIN rev USING(customer_id)
LEFT JOIN prod USING(customer_id) LEFT JOIN seg_customers_360 cc USING(customer_id)
""")
print('клиентов в окне:', len(feat))
feat.head(3)

In [ ]:
# 29 клиентов без исходящих операций (только входящие) -> доли не определены. Убираем из кластеризации.
before = len(feat)
feat = feat[feat['out_cnt'] > 0].reset_index(drop=True)
print(f'убрано без исходящих: {before-len(feat)}; осталось: {len(feat)}')

## 4. Разведка признаков
Денежные и счётные признаки сильно скошены — логарифмируем перед масштабированием.

In [ ]:
num = ['txn_per_month','atm_share','ecom_share','avg_ticket','avg_balance','inflow_share']
feat[num].describe().round(2)

In [ ]:
fig, ax = plt.subplots(2,3, figsize=(13,6))
for a, col in zip(ax.ravel(), num):
    a.hist(feat[col], bins=50); a.set_title(col)
plt.tight_layout(); plt.show()

## 5. Предобработка
log1p для скошенных, затем стандартизация. Выручку в оси НЕ берём — она выход, её меряем по сегментам.

In [ ]:
feat['l_txn_pm']  = np.log1p(feat['txn_per_month'])
feat['l_ticket']  = np.log1p(feat['avg_ticket'])
feat['l_balance'] = np.log1p(feat['avg_balance'])

FEATURES = ['l_txn_pm','atm_share','ecom_share','l_ticket','l_balance','inflow_share']
scaler = StandardScaler().fit(feat[FEATURES])
X = scaler.transform(feat[FEATURES])
X.shape

## 6. Выбор числа кластеров
Локоть по инерции + силуэт. Смотрим k=2..8.

In [ ]:
ks = range(2,9); inertia=[]; sil=[]
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(X, km.labels_, sample_size=5000, random_state=0))
fig, ax = plt.subplots(1,2, figsize=(11,3.2))
ax[0].plot(list(ks), inertia, 'o-'); ax[0].set_title('Инерция (локоть)'); ax[0].set_xlabel('k')
ax[1].plot(list(ks), sil, 'o-');     ax[1].set_title('Силуэт');          ax[1].set_xlabel('k')
plt.tight_layout(); plt.show()
pd.DataFrame({'k':list(ks),'inertia':np.round(inertia),'silhouette':np.round(sil,3)})

**Вывод:** максимум силуэта и локоть — на **k=4**.

In [ ]:
K = 4
km = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20).fit(X)
feat['seg'] = km.labels_
print('силуэт k=4:', round(silhouette_score(X, km.labels_, sample_size=5000, random_state=0), 3))
feat['seg'].value_counts().sort_index()

## 7. Профили сегментов

In [ ]:
prof = feat.groupby('seg').agg(
    n            =('customer_id','size'),
    txn_per_month=('txn_per_month','median'),
    atm_share    =('atm_share','median'),
    ecom_share   =('ecom_share','median'),
    avg_ticket   =('avg_ticket','median'),
    avg_balance  =('avg_balance','median'),
    net_rev_med  =('net_rev_12mo','median'),
    n_products   =('n_products','median'),
).round(2)
prof['rev_total_bn'] = (feat.groupby('seg')['net_rev_12mo'].sum()/1e9).round(2)
prof['cust_%']       = (prof['n']/len(feat)*100).round(1)
prof['rev_%']        = (feat.groupby('seg')['net_rev_12mo'].sum()/feat['net_rev_12mo'].sum()*100).round(1)
prof

In [ ]:
# именуем по профилю
NAMES = {}
for s, r in prof.iterrows():
    if   r['avg_balance'] > 1e7:                         NAMES[s] = 'Wealthy Depositors'
    elif r['txn_per_month'] > 35 and r['ecom_share']>.3: NAMES[s] = 'Digital Spenders'
    elif r['net_rev_med']  < 3e5:                        NAMES[s] = 'Cash / Passive'
    else:                                                NAMES[s] = 'Mass Active'
feat['type'] = feat['seg'].map(NAMES)
print(NAMES)

In [ ]:
fig, ax = plt.subplots(1,3, figsize=(13,3.4))
order = feat.groupby('type')['net_rev_12mo'].sum().sort_values().index
feat.groupby('type')['customer_id'].size().reindex(order).plot.barh(ax=ax[0], title='Размер сегмента')
(feat.groupby('type')['net_rev_12mo'].sum()/1e9).reindex(order).plot.barh(ax=ax[1], title='Выручка, млрд UZS')
feat.groupby('type')['atm_share'].median().reindex(order).plot.barh(ax=ax[2], title='Медианная доля ATM')
plt.tight_layout(); plt.show()

## 8. Устойчивость (ARI относительно базовой модели)

Проверяем инвариантность к: сиду, половине выборки, удалению последнего месяца, смене набора признаков.

In [ ]:
base = pd.Series(km.labels_, index=feat['customer_id'])
rows = []

# a) другой random_state
for rs in [1, 7, 2024]:
    lab = KMeans(n_clusters=K, random_state=rs, n_init=20).fit(X).labels_
    rows.append((f'random_state={rs}', adjusted_rand_score(base, lab)))

# b) половина выборки (переобучаем на ней, сравниваем на её же точках)
idx = np.random.RandomState(0).choice(len(feat), len(feat)//2, replace=False)
Xs  = StandardScaler().fit_transform(feat.iloc[idx][FEATURES])
lab = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20).fit(Xs).labels_
rows.append(('половина выборки', adjusted_rand_score(base.iloc[idx], lab)))

# d) изменённый набор признаков
F2  = ['l_txn_pm','atm_share','ecom_share','l_ticket','inflow_share','n_products','age']
Xf2 = StandardScaler().fit_transform(feat[F2])
lab = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20).fit(Xf2).labels_
rows.append(('изменённые признаки', adjusted_rand_score(base, lab)))

pd.DataFrame(rows, columns=['возмущение','ARI']).round(3)

In [ ]:
# c) данные без последнего месяца: пересчитываем фичи по окну без дек-2025 и рекластеризуем
nl = sql(f"""
WITH win AS (SELECT * FROM seg_transactions WHERE txn_ts>=DATE '{WIN_START}' AND txn_ts<DATE '2025-12-01'),
tx AS (SELECT customer_id, count(*) txn_cnt,
   count(DISTINCT date_trunc('month',txn_ts)) am,
   count(*) FILTER(WHERE direction='out') out_cnt,
   count(*) FILTER(WHERE direction='out' AND channel='atm') atm_cnt,
   count(*) FILTER(WHERE direction='out' AND channel='ecom') ecom_cnt,
   count(*) FILTER(WHERE direction='in' AND channel='p2p') p2p_in_cnt,
   sum(amount_uzs) FILTER(WHERE direction='out') out_amt FROM win GROUP BY 1),
bal AS (SELECT customer_id, avg(avg_balance_uzs) avg_balance FROM seg_balances
        WHERE month_start>=DATE '{WIN_START}' AND month_start<DATE '2025-12-01' GROUP BY 1)
SELECT customer_id,
   ln(1+tx.txn_cnt::double precision/NULLIF(am,0)) l_txn_pm,
   tx.atm_cnt::double precision/NULLIF(out_cnt,0)  atm_share,
   tx.ecom_cnt::double precision/NULLIF(out_cnt,0) ecom_share,
   ln(1+tx.out_amt::double precision/NULLIF(out_cnt,0)) l_ticket,
   ln(1+COALESCE(bal.avg_balance,0))    l_balance,
   tx.p2p_in_cnt::double precision/NULLIF(tx.txn_cnt,0) inflow_share
FROM tx LEFT JOIN bal USING(customer_id) WHERE out_cnt>0""").set_index('customer_id')

common = base.index.intersection(nl.index)
lab = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20).fit(
        StandardScaler().fit_transform(nl[FEATURES])).labels_
lab = pd.Series(lab, index=nl.index)
print('без последнего месяца  ARI =',
      round(adjusted_rand_score(base.loc[common], lab.loc[common]), 3),
      f'(общих клиентов {len(common)})')

**Итог устойчивости:** к сиду и подвыборке ~1.0, к удалению месяца ~0.97, к смене признаков ~0.86.
Структура держится на поведении, а не на одном признаке.

## 9. Сравнение с бэйслайнами

(1) квартили оборота, (2) готовый `crm_segment` из старой витрины.
ARI показывает, насколько это другое разбиение; eta² — сколько дисперсии поведения объясняет каждое.

In [ ]:
def eta2(groups, y):
    y = np.asarray(y, float); gm = y.mean(); sst = ((y-gm)**2).sum()
    ssb = sum((groups==g).sum()*((y[groups==g].mean()-gm)**2) for g in np.unique(groups))
    return ssb/sst if sst>0 else 0.0

qt = pd.qcut(feat['turnover_total'], 4, labels=[0,1,2,3]).astype(int).values
cr = feat['crm_segment'].astype('category').cat.codes.values
cl = feat['seg'].values

print('ARI(кластеры, квартили оборота):', round(adjusted_rand_score(cl, qt), 3))
print('ARI(кластеры, crm_segment):     ', round(adjusted_rand_score(cl, cr), 3))
print()
cmp = pd.DataFrame({
    'eta2_кластеры': [eta2(cl, feat[c]) for c in ['atm_share','ecom_share','avg_ticket','avg_balance','net_rev_12mo']],
    'eta2_квартили': [eta2(qt, feat[c]) for c in ['atm_share','ecom_share','avg_ticket','avg_balance','net_rev_12mo']],
    'eta2_crm':      [eta2(cr, feat[c]) for c in ['atm_share','ecom_share','avg_ticket','avg_balance','net_rev_12mo']],
}, index=['atm_share','ecom_share','avg_ticket','avg_balance','net_rev_12mo']).round(3)
cmp

**Вывод.** Кластеры — другое разбиение, чем квартили (ARI≈0.25), и в разы лучше разделяют поведение
(наличные, онлайн, флоат). По самой выручке квартили чуть сильнее — они на обороте и построены, это честно.
`crm_segment` бесполезен: ARI≈0 и eta²≈0 по всем осям (все A/B/C/D одинаковы) — как основу не берём.

## 10. Скоринг нового клиента (nearest-centroid) + проверка вне обучения

Правило: посчитать те же 6 признаков → стандартизовать сохранённым scaler → отнести к ближайшему центроиду.
Проверяем на отложенных 20%: обучаемся на 80%, назначаем holdout по центроидам, сравниваем с полной перекластеризацией.

In [ ]:
def assign_segment(df_new, scaler, model):
    """df_new с колонками FEATURES -> метки сегментов по ближайшему центроиду."""
    return model.predict(scaler.transform(df_new[FEATURES]))

# holdout-проверка
rng = np.random.RandomState(1)
te = rng.choice(len(feat), int(len(feat)*0.2), replace=False)
tr = np.setdiff1d(np.arange(len(feat)), te)

sc_tr = StandardScaler().fit(feat.iloc[tr][FEATURES])
km_tr = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=20).fit(sc_tr.transform(feat.iloc[tr][FEATURES]))
te_lab = assign_segment(feat.iloc[te], sc_tr, km_tr)

full = base.reset_index(drop=True).values  # базовые метки на всех
print('ARI(правило на holdout, полная перекластеризация) =',
      round(adjusted_rand_score(full[te], te_lab), 3))

# воспроизводимость профиля train vs holdout
tr_df = feat.iloc[tr].assign(seg2=km_tr.labels_)
te_df = feat.iloc[te].assign(seg2=te_lab)
pd.DataFrame({
    'train_rev_med':   tr_df.groupby('seg2')['net_rev_12mo'].median().round(0),
    'holdout_rev_med': te_df.groupby('seg2')['net_rev_12mo'].median().round(0),
    'train_atm':       tr_df.groupby('seg2')['atm_share'].median().round(2),
    'holdout_atm':     te_df.groupby('seg2')['atm_share'].median().round(2),
})

> **Cold start:** у клиента без истории признаков ещё нет — первые 1–2 месяца назначаем дефолт (массовый сегмент) или прогноз по демографии/первому депозиту, затем переназначаем по правилу.

## 11. Выгрузка результатов

In [ ]:
out = feat[['customer_id','type','seg','net_rev_12mo','atm_share','ecom_share',
            'avg_balance','txn_per_month']].copy()
out.to_csv('customer_segments.csv', index=False)

import joblib
joblib.dump({'scaler':scaler, 'model':km, 'features':FEATURES, 'names':NAMES}, 'kmeans_model.joblib')
print('сохранено: customer_segments.csv, kmeans_model.joblib')
out.head()

## Итоги для бизнеса

4 типа с разной ценностью и стоимостью обслуживания при **едином** тарифе:

| Тип | Доля клиентов | Доля выручки | Что делать |
|---|---|---|---|
| Cash / Passive | ~38% | ~6% | цифровой онбординг, пересмотр субсидии бесплатного ATM |
| Digital Spenders | ~23% | ~17% | кэшбэк, маркетплейс, credit_line / instalment |
| Mass Active | ~24% | ~43% | сдвиг наличных в POS (дорогой ATM-кост), кросс-сейл депозитов |
| Wealthy Depositors | ~15% | ~34% | премиум-тариф, депозиты/инвест, удержание |

**Решение:** заменить единый `standard` на 3–4 дифференцированных пакета и перераспределить
субсидию ATM-лимита с низкоценных кассовых клиентов на удержание вкладчиков и рост цифровых.